# Limpieza de Datos

# 1. Transformación fechas a signos y eliminar filas
El objetivo es transformar las fechas de nacimiento a signos zodiacales, los datos que no posean la fecha de nacimiento generan el valor desconocido, el cual debe eliminarse al no ser utilizable en el dataset

In [ ]:
import pandas as pd
import os

# Diccionario con las fechas de los signos zodiacales (NO MODIFICAR)
signo_fechas = {
    "Aries": ((3, 21), (4, 19)),
    "Tauro": ((4, 20), (5, 20)),
    "Géminis": ((5, 21), (6, 21)),
    "Cáncer": ((6, 21), (7, 22)),
    "Leo": ((7, 23), (8, 22)),
    "Virgo": ((8, 23), (9, 22)),
    "Libra": ((9, 23), (10, 23)),
    "Escorpio": ((10, 24), (11, 21)),
    "Sagitario": ((11, 22), (12, 21)),
    "Capricornio": ((12, 22), (1, 19)),
    "Acuario": ((1, 20), (2, 18)),
    "Piscis": ((2, 19), (3, 20)),
}

# Función obtener_signo (NO MODIFICAR)
def obtener_signo(dia, mes):
    """Calcula el signo zodiacal a partir del día y mes."""
    for signo, fechas in signo_fechas.items():
        if ((mes == fechas[0][0] and dia >= fechas[0][1]) or
            (mes == fechas[1][0] and dia <= fechas[1][1])):
            return signo
    if (mes == 12 and dia >= 22) or (mes == 1 and dia <= 19):
        return "Capricornio"
    return 'Desconocido'

try:
    # =============================================================================
    # Configuración de rutas
    # =============================================================================
    archivo_original = '../1_data_processed/v0_base_de_datos_unificada.csv'
    archivo_salida = '../1_data_processed/v1_base_filas_validas.csv'
    archivo_eliminadas = '../4_results/v1_filas_eliminadas.csv'

    # Asegurar que exista la carpeta de salida
    os.makedirs(os.path.dirname(archivo_salida), exist_ok=True)

    print("Cargando el archivo completo...")
    df = pd.read_csv(archivo_original, low_memory=False)
    total_filas_original = df.shape[0]
    total_cols_original = df.shape[1]

    # =============================================================================
    # 1. TRANSFORMACIÓN: FECHA -> SIGNO
    # =============================================================================
    print("--- Iniciando Transformación FECHA_NACIMIENTO -> SIGNO_ZODIACAL ---")

    if 'FECHA_NACIMIENTO' not in df.columns:
        raise KeyError("No existe la columna 'FECHA_NACIMIENTO' en el CSV original.")

    # Convertir FECHA_NACIMIENTO a datetime (inválidos -> NaT)
    df['FECHA_NACIMIENTO'] = pd.to_datetime(df['FECHA_NACIMIENTO'], errors='coerce')

    # Crear SIGNO_ZODIACAL
    df['SIGNO_ZODIACAL'] = df.apply(
        lambda row: obtener_signo(row['FECHA_NACIMIENTO'].day, row['FECHA_NACIMIENTO'].month)
        if pd.notna(row['FECHA_NACIMIENTO']) else 'Desconocido',
        axis=1
    )

    # =============================================================================
    # 2. LIMPIEZA DE FILAS: eliminar filas sin fecha válida o sin signo
    # =============================================================================
    print("--- Eliminando filas sin fecha válida o con signo 'Desconocido' ---")

    mask_invalidas = df['FECHA_NACIMIENTO'].isna() | (df['SIGNO_ZODIACAL'] == 'Desconocido')

    df_eliminadas = df.loc[mask_invalidas].copy()
    df_validas = df.loc[~mask_invalidas].copy()

    print(f"Filas originales: {total_filas_original:,}")
    print(f"Filas eliminadas: {df_eliminadas.shape[0]:,}")
    print(f"Filas válidas: {df_validas.shape[0]:,}")

    # Guardar eliminadas (opcional, pero recomendado)
    if df_eliminadas.shape[0] > 0:
        df_eliminadas.to_csv(archivo_eliminadas, index=False)
        print(f"✅ Filas eliminadas guardadas en: {archivo_eliminadas}")
        
    if "FECHA_NACIMIENTO" in df_validas.columns:
        df_validas.drop(columns=["FECHA_NACIMIENTO"], inplace=True)

    # Guardar dataset válido completo (NO se elimina ninguna columna)
    df_validas.to_csv(archivo_salida, index=False)

    print("\n" + "=" * 60)
    print("¡PROCESO COMPLETADO!")
    print(f"Archivo de salida: {archivo_salida}")
    print(f"Columnas mantenidas: {df_validas.shape[1]} (originales: {total_cols_original})")
    print("=" * 60)

except FileNotFoundError:
    print(f"Error: El archivo '{archivo_original}' no fue encontrado.")
except KeyError as e:
    print(f"Error: {str(e)}")
except Exception as e:
    print(f"Ocurrió un error inesperado: {str(e)}")

Cargando el archivo completo...
--- Iniciando Transformación FECHA_NACIMIENTO -> SIGNO_ZODIACAL ---
--- Eliminando filas sin fecha válida o con signo 'Desconocido' ---
Filas originales: 5,808,535
Filas eliminadas: 37
Filas válidas: 5,808,498
✅ Filas eliminadas guardadas en: ../1_data_processed/v1_filas_eliminadas.csv

¡PROCESO COMPLETADO!
Archivo de salida: ../1_data_processed/v1_base_filas_validas.csv
Columnas mantenidas: 130 (originales: 130)


# Generar Archivo solo con las columnas necesarias

In [6]:
import pandas as pd
import os

archivo_csv = "../1_data_processed/v1_base_filas_validas.csv"
archivo_csv_slim = "../1_data_processed/v1_base_filas_validas_slim.csv"

os.makedirs("../1_data_processed/", exist_ok=True)

N_DIAG = 35
N_PROC = 30

diag_cols = [f"DIAGNOSTICO{i}" for i in range(1, N_DIAG + 1)]
proc_cols = [f"PROCEDIMIENTO{i}" for i in range(1, N_PROC + 1)]

cols_needed = ["SIGNO_ZODIACAL", "ESPECIALIDAD_MEDICA", "FECHA_INGRESO", "FECHAALTA"] + diag_cols + proc_cols
cols_needed_set = set(cols_needed)

chunksize = 250_000  # baja a 100_000 si aún pesa

# Si existe, lo sobreescribimos
if os.path.exists(archivo_csv_slim):
    os.remove(archivo_csv_slim)

total_rows = 0
first = True

for chunk in pd.read_csv(
    archivo_csv,
    usecols=lambda c: c in cols_needed_set,
    low_memory=False,
    chunksize=chunksize
):
    # Escribe por partes (append). No acumulamos en RAM.
    chunk.to_csv(archivo_csv_slim, mode="a", index=False, header=first)
    first = False

    total_rows += len(chunk)
    print(f"✅ Chunk escrito: {chunk.shape} | acumulado filas: {total_rows:,}")

print("✅ CSV slim guardado en:", archivo_csv_slim)

✅ Chunk escrito: (250000, 69) | acumulado filas: 250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 1,750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 2,750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,250,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,500,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 3,750,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 4,000,000
✅ Chunk escrito: (250000, 69) | acumulado filas: 4,250,000
✅ C

# 2. Filtro de Columnas
Columnas a mantener asociadas a diagnósticos

In [3]:
import pandas as pd
import numpy as np
import os
import re
import unicodedata

from sklearn.feature_selection import chi2
from sklearn.preprocessing import LabelEncoder

from scipy import sparse

# =========================
# CONFIGURACIÓN
# =========================
archivo_entrada = "../1_data_processed/v1_base_filas_validas_slim.csv"

archivo_pre_chi = "../1_data_processed/v2_dataset_pre_chi.csv"
archivo_resultados_chi = "../4_results/v3_chi2_resultados.csv"

os.makedirs("../1_data_processed/", exist_ok=True)
os.makedirs("../4_results/", exist_ok=True)

TARGET = "SIGNO_ZODIACAL"

SIGNOS_FIJOS = [
    "Acuario", "Aries", "Capricornio", "Cáncer",
    "Escorpio", "Géminis", "Leo", "Libra",
    "Piscis", "Sagitario", "Tauro", "Virgo"
]

N_DIAG = 35
N_PROC = 30

# =========================
# HELPERS
# =========================
def cie10_letra(codigo):
    if pd.isna(codigo):
        return "NA"
    s = str(codigo).strip().upper()
    if s == "" or s in {"NAN", "NONE"}:
        return "NA"
    m = re.search(r"[A-Z]", s)
    return m.group(0) if m else "OTROS"

def normalizar_signo(s):
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-z0-9_]", "", s)
    return s

# =========================
# PASO 0: CARGAR (solo columnas necesarias)
# =========================
diag_cols = [f"DIAGNOSTICO{i}" for i in range(1, N_DIAG + 1)]

cols_needed = [TARGET, "ESPECIALIDAD_MEDICA", "FECHA_INGRESO", "FECHAALTA"] + diag_cols
cols_needed_set = set(cols_needed)

df = pd.read_csv(archivo_entrada, usecols=lambda c: c in cols_needed_set, low_memory=False)
print("Shape inicial:", df.shape)

missing_min = [c for c in [TARGET, "ESPECIALIDAD_MEDICA", "FECHA_INGRESO", "FECHAALTA"] if c not in df.columns]
if missing_min:
    raise KeyError(f"Faltan columnas mínimas en el CSV slim: {missing_min}")

# =========================
# PASO 1: ESTANCIA_DIAS
# =========================
df["FECHA_INGRESO"] = pd.to_datetime(df["FECHA_INGRESO"], errors="coerce")
df["FECHAALTA"] = pd.to_datetime(df["FECHAALTA"], errors="coerce")

df["ESTANCIA_DIAS"] = (df["FECHAALTA"] - df["FECHA_INGRESO"]).dt.days
df["ESTANCIA_DIAS"] = df["ESTANCIA_DIAS"].fillna(0).astype(np.int32)
df.loc[df["ESTANCIA_DIAS"] < 0, "ESTANCIA_DIAS"] = 0

# liberar fechas ASAP
df.drop(columns=["FECHA_INGRESO", "FECHAALTA"], inplace=True)

# =========================
# PASO 2: FILTRO BASE
# =========================
df = df[[TARGET, "ESPECIALIDAD_MEDICA", "ESTANCIA_DIAS"] + diag_cols].copy()
print("Shape después del filtro base:", df.shape)

# =========================
# PASO 3: TRANSFORMACIONES
# =========================
for c in diag_cols:
    df[c] = df[c].apply(cie10_letra)


print("✅ Transformación diagnósticos completada.")

# =========================
# PASO 4: ONE-HOT SPARSE
# =========================
cat_cols = ["ESPECIALIDAD_MEDICA"] + diag_cols

for c in cat_cols:
    df[c] = df[c].astype("category")

X_cat = pd.get_dummies(df[cat_cols], drop_first=False, sparse=True)
print("✅ One-hot (sparse) listo:", X_cat.shape)

# Convertir a CSR float32 ANTES de chi2
X_csr = X_cat.sparse.to_coo().tocsr().astype(np.float32)

# =========================
# PASO 5: CHI² (con sparse float32)
# =========================
le = LabelEncoder()
y_encoded = le.fit_transform(df[TARGET].astype("object"))

chi_scores, p_values = chi2(X_csr, y_encoded)

resultados = pd.DataFrame({
    "variable": X_cat.columns.astype(str),
    "chi2": chi_scores,
    "p_value": p_values
}).sort_values("chi2", ascending=False)

resultados.to_csv(archivo_resultados_chi, index=False)
print("✅ Resultados chi guardados en:", archivo_resultados_chi)

# =========================
# PASO 6: BINARIZAR TARGET (ONE-VS-REST)
# =========================
Y_bin = pd.DataFrame(index=df.index)
for signo in SIGNOS_FIJOS:
    colname = f"signo_zodiacal_{normalizar_signo(signo)}"
    Y_bin[colname] = (df[TARGET] == signo).astype(np.uint8)

print("✅ Targets binarios creados:", list(Y_bin.columns))

# =========================
# GUARDAR SOLO PRE_CHI (SIN POST_CHI)
# =========================
df_pre_chi = pd.concat([X_cat, df["ESTANCIA_DIAS"], Y_bin], axis=1)
df_pre_chi.to_csv(archivo_pre_chi, index=False)
print("✅ Dataset PRE_CHI guardado:", df_pre_chi.shape, "->", archivo_pre_chi)

print("✅ Listo: se generó PRE_CHI y resultados chi². (POST_CHI NO se genera en este script)")

Shape inicial: (5808498, 39)
Shape después del filtro base: (5808498, 38)
✅ Transformación diagnósticos completada.
✅ One-hot (sparse) listo: (5808498, 1044)
✅ Resultados chi guardados en: ../4_results/v3_chi2_resultados.csv
✅ Targets binarios creados: ['signo_zodiacal_acuario', 'signo_zodiacal_aries', 'signo_zodiacal_capricornio', 'signo_zodiacal_cancer', 'signo_zodiacal_escorpio', 'signo_zodiacal_geminis', 'signo_zodiacal_leo', 'signo_zodiacal_libra', 'signo_zodiacal_piscis', 'signo_zodiacal_sagitario', 'signo_zodiacal_tauro', 'signo_zodiacal_virgo']


KeyboardInterrupt: 

In [1]:
import pandas as pd
import os

archivo_pre_chi = "../1_data_processed/v2_dataset_pre_chi.csv"
archivo_chi = "../4_results/v3_chi2_resultados.csv"
archivo_post_chi = "../1_data_processed/v3_dataset_post_chi.csv"

os.makedirs("../1_data_processed/", exist_ok=True)

# =========================
# Cargar resultados chi²
# =========================
df_chi = pd.read_csv(archivo_chi)

vars_significativas = df_chi.loc[
    df_chi["p_value"] < 0.05,
    "variable"
].tolist()

print("Variables significativas:", len(vars_significativas))

# =========================
# Identificar columnas target
# =========================
targets = [
    "signo_zodiacal_acuario",
    "signo_zodiacal_aries",
    "signo_zodiacal_capricornio",
    "signo_zodiacal_cancer",
    "signo_zodiacal_escorpio",
    "signo_zodiacal_geminis",
    "signo_zodiacal_leo",
    "signo_zodiacal_libra",
    "signo_zodiacal_piscis",
    "signo_zodiacal_sagitario",
    "signo_zodiacal_tauro",
    "signo_zodiacal_virgo"
]

extra_cols = ["ESTANCIA_DIAS"] + targets

cols_needed = vars_significativas + extra_cols
cols_needed = list(set(cols_needed))

print("Columnas a cargar:", len(cols_needed))

# =========================
# Generar POST_CHI en chunks
# =========================
chunksize = 200000

first = True

for chunk in pd.read_csv(
    archivo_pre_chi,
    usecols=lambda c: c in cols_needed,
    chunksize=chunksize
):

    if first:
        chunk.to_csv(archivo_post_chi, index=False, mode="w")
        first = False
    else:
        chunk.to_csv(archivo_post_chi, index=False, mode="a", header=False)

    print("Chunk procesado:", chunk.shape)

print("✅ Dataset POST_CHI generado:", archivo_post_chi)

Variables significativas: 742
Columnas a cargar: 755
Chunk procesado: (200000, 755)
Chunk procesado: (200000, 755)
Chunk procesado: (44106, 755)
✅ Dataset POST_CHI generado: ../1_data_processed/v3_dataset_post_chi.csv


In [4]:
import pandas as pd

archivo = "../1_data_processed/v3_dataset_post_chi.csv"

total = 0
for chunk in pd.read_csv(archivo, chunksize=500_000, usecols=[0]):  # lee solo 1 columna
    total += len(chunk)

print("Filas reales en el CSV slim:", total)

Filas reales en el CSV slim: 444106
